In [121]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# from langchain_core.chatopenai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableSequence
import os
from dotenv import load_dotenv


load_dotenv()



# generation = input("Enter your generation (e.g., GenZ, Millennial, GenX): ")

True

### ***Chains with Custom Runnables***

##### Chain 1

In [122]:
# Task 1 - Prompt Template Creation
user_input = input("Enter a movie name: ")
prompt_template = ChatPromptTemplate.from_messages([("user", "{topic}"),
("system", "You are a movie summary generator who answers in a concise and witty manner.")])

In [123]:
# Task 2 - Prompt Finalization.

ready_prompt = prompt_template.invoke({"topic": user_input, "generation": "millennial"})
llm_openai= init_chat_model(model=os.environ.get("LLM_MODEL"), temperature = 0.7)

In [124]:
# Task 3 - String Output Parser Creation

str_parse = StrOutputParser()

In [125]:
def dictionary_parser(text:str)->dict:

    return {"text": text}

In [126]:
prompt_post = ChatPromptTemplate.from_messages([("user", "Create a movie review from the summary {text}"),
("system", "You are a movie critic on Twitter.")])

In [127]:
linkedin_chain =  dictionary_parser | prompt_post | llm_openai | str_parse

##### Chain 2

In [128]:
def insta_chain(text):

    # print(text)

    text = text
    #Task 1 - Prompt Template Creation
    insta_prompt = ChatPromptTemplate.from_messages([("user", "Create an Instagram movie review from the summary {text}"),
    ("system", "You are a social media influencer. You need to post on Instagram based on the text that is provided to you.")])

    llm_openai= init_chat_model(model=os.environ.get("LLM_MODEL"), temperature = 0.7)

    str_parse = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parse

    result = chain_insta.invoke({"text": text})

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

#### ***Orchestrator***

In [129]:
# Chain with custom task

final_chain = (prompt_template | llm_openai | str_parse |
RunnableParallel(branches = {"linkedin": linkedin_chain, "instagram": insta_chain_runnable}))

# final_chain.invoke({"topic": user_input})

In [130]:
from pydantic import BaseModel

class Posts(BaseModel):
    linkedin: str
    instagram: str

In [ ]:
def beautify(final_response: dict):
    
    linkedin_response = final_response["branches"]["linkedin"]
    instagram_response = final_response["branches"]['instagram']

    beautify_prompt = ChatPromptTemplate.from_messages([("system", "You are social media assistant. You clean all the posts and cross check it."),
    ("user", "LinkedIn post:\n{linkedin}\n\nInstagram post:\n{instagram}")])

    llm_openai= init_chat_model(model=os.environ.get("LLM_MODEL"), temperature = 0.7)

    beautify_chain = beautify_prompt | llm_openai.with_structured_output(Posts)

    result = beautify_chain.invoke({"linkedin": linkedin_response, "instagram": instagram_response})
    
    return result.model_dump()

beautify_runnable = RunnableLambda(beautify)

beautify_chain_final = final_chain | beautify_runnable

beautify_chain_final.invoke(user_input)